In [54]:
import sys
sys.path.append("..")
import pandas as pd
from src.data import load_tables

tables = load_tables("../data/raw")
failures = tables["failures"]


In [55]:
machine_ids = tables["machines"]["machineID"]
days = pd.date_range("2015-01-01", "2015-12-31", freq="D") + pd.Timedelta(hours=6)

skeleton = pd.MultiIndex.from_product(
    [machine_ids, days], names=["machineID", "prediction_time"]
).to_frame(index=False)

print(skeleton.shape)    # expect (36500, 2)
skeleton.head()

(36500, 2)


,machineID,prediction_time
0,1,2015-01-01 06:00:00
1,1,2015-01-02 06:00:00
2,1,2015-01-03 06:00:00
3,1,2015-01-04 06:00:00
4,1,2015-01-05 06:00:00


In [56]:
HORIZON = pd.Timedelta(hours=24)

# Failures on the same machine at the same time count once for a yes/no target
events = failures[["machineID", "datetime"]].drop_duplicates()

# Pair each machine-day with all failures of that same machine
pairs = skeleton.merge(events, on="machineID", how="left")

# Keep the pairs where the failure is inside (T, T + 24h]
in_window = (
    (pairs["datetime"] > pairs["prediction_time"])
    & (pairs["datetime"] <= pairs["prediction_time"] + HORIZON)
)
positive = pairs.loc[in_window, ["machineID", "prediction_time"]].drop_duplicates()
positive["failure_next_24h"] = 1

# Attach to the skeleton; rows with no match are the 0s
df = skeleton.merge(positive, on=["machineID", "prediction_time"], how="left")
df["failure_next_24h"] = df["failure_next_24h"].fillna(0).astype(int)

In [57]:
print("rows:", len(df))
print("positive rows:", df["failure_next_24h"].sum())
print("positive rate:", df["failure_next_24h"].mean().round(4))
print("failure events (deduplicated):", len(failures[["machineID", "datetime"]].drop_duplicates()))

rows: 36500
positive rows: 718
positive rate: 0.0197
failure events (deduplicated): 719


In [58]:
f = failures.iloc[10]
print(f)
df[(df["machineID"] == f["machineID"])
   & (df["prediction_time"].between(f["datetime"] - pd.Timedelta(days=2),
                                    f["datetime"] + pd.Timedelta(days=1)))]

datetime     2015-12-29 06:00:00
machineID                      2
failure                    comp2
Name: 10, dtype: object


,machineID,prediction_time,failure_next_24h
725,2,2015-12-27 06:00:00,0
726,2,2015-12-28 06:00:00,1
727,2,2015-12-29 06:00:00,0
728,2,2015-12-30 06:00:00,0


In [59]:
events = failures[["machineID", "datetime"]].drop_duplicates()

# Which rows are claimed by more than one failure event?
pairs = skeleton.merge(events, on="machineID", how="left")
in_window = (pairs["datetime"] > pairs["prediction_time"]) & \
            (pairs["datetime"] <= pairs["prediction_time"] + HORIZON)
hits = pairs[in_window].groupby(["machineID", "prediction_time"]).size()
print(hits[hits > 1])

machineID  prediction_time    
73         2015-01-01 06:00:00    2
dtype: int64


In [60]:
tel = tables["telemetry"]
SENSORS = ["volt", "rotate", "pressure", "vibration"]

tel = tel.sort_values(["machineID", "datetime"])

rolled = (
    tel.set_index("datetime")
       .groupby("machineID")[SENSORS]
       .rolling("24h")              # window = (t - 24h, t]  -> never looks past t
       .agg(["mean", "std"])
)
rolled.columns = [f"{s}_{stat}_24h" for s, stat in rolled.columns]
rolled = rolled.reset_index()       # machineID, datetime, + 8 feature columns
rolled.head()

,machineID,datetime,volt_mean_24h,volt_std_24h,rotate_mean_24h,rotate_std_24h,pressure_mean_24h,pressure_std_24h,vibration_mean_24h,vibration_std_24h
0,1,2015-01-01 06:00:00,176.217853,NaN,418.504078,NaN,113.077935,NaN,45.087686,NaN
1,1,2015-01-01 07:00:00,169.548538,9.431836,410.625784,11.141591,104.269230,12.457390,44.250829,1.183494
2,1,2015-01-01 08:00:00,170.028993,6.721032,449.533798,67.849599,94.592122,18.934956,40.893502,5.874970
3,1,2015-01-01 09:00:00,168.137453,6.665324,423.687682,75.770259,98.256232,17.109194,40.950662,4.798255
4,1,2015-01-01 10:00:00,166.031967,7.448844,426.025520,65.826868,100.982315,16.021893,37.958632,7.875828


In [61]:
features = df.merge(
    rolled,
    left_on=["machineID", "prediction_time"],
    right_on=["machineID", "datetime"],
    how="left",
).drop(columns="datetime")

print(features.shape)                     # expect (36500, 11)
print(features.isna().sum())              # which columns have gaps?
features.head()

(36500, 11)
machineID               0
prediction_time         0
failure_next_24h        0
volt_mean_24h           0
volt_std_24h          100
rotate_mean_24h         0
rotate_std_24h        100
pressure_mean_24h       0
pressure_std_24h      100
vibration_mean_24h      0
vibration_std_24h     100
dtype: int64


,machineID,prediction_time,failure_next_24h,volt_mean_24h,volt_std_24h,rotate_mean_24h,rotate_std_24h,pressure_mean_24h,pressure_std_24h,vibration_mean_24h,vibration_std_24h
0,1,2015-01-01 06:00:00,0,176.217853,NaN,418.504078,NaN,113.077935,NaN,45.087686,NaN
1,1,2015-01-02 06:00:00,0,170.614862,12.519402,446.364859,48.385076,96.849785,10.171540,39.736826,6.163231
2,1,2015-01-03 06:00:00,0,169.533156,13.995465,454.785072,40.843882,100.050567,11.036546,39.449734,5.561553
3,1,2015-01-04 06:00:00,1,172.042428,12.848646,450.418764,45.090576,100.284484,9.846662,52.153213,5.888262
4,1,2015-01-05 06:00:00,0,171.929104,21.298322,443.448775,36.054002,98.675590,9.885781,51.780445,5.246894


In [62]:
row = features.iloc[1]                      # machine 1, 2015-01-03 06:00
T = row["prediction_time"]
t = tel[(tel["machineID"] == row["machineID"])

        & (tel["datetime"] <= T)]

print(len(t))                               # expect 24 readings
print(t["volt"].mean(), row["volt_mean_24h"])   # should match

25
170.8389815213688 170.6148618757748


In [63]:
# Rows with less than 24h of telemetry history (first day of data) are dropped
features = features[features["prediction_time"] >= "2015-01-02"].reset_index(drop=True)

print(features.shape)                       # expect (36400, 11)
print(features.isna().sum().sum())          # expect 0
print(features["failure_next_24h"].sum())   # expect 717

(36400, 11)
0
699


In [64]:
err = tables["errors"]
ERRORS = ["error1", "error2", "error3", "error4", "error5"]

counts = (err.groupby(["machineID", "datetime", "errorID"]).size()
             .unstack(fill_value=0)
             .reindex(columns=ERRORS, fill_value=0))

grid = pd.MultiIndex.from_frame(tel[["machineID", "datetime"]])   # every machine-hour
counts_hourly = counts.reindex(grid, fill_value=0)

# Check: no error should be lost by moving to the grid
print(counts.values.sum(), counts_hourly.values.sum())   # must be equal

3919 3919


In [65]:
rolled_err = (counts_hourly.reset_index()
                .set_index("datetime")
                .groupby("machineID")[ERRORS]
                .rolling("24h").sum())
rolled_err.columns = [f"{e}_count_24h" for e in ERRORS]
rolled_err = rolled_err.reset_index()

features = features.merge(
    rolled_err,
    left_on=["machineID", "prediction_time"],
    right_on=["machineID", "datetime"],
    how="left",
).drop(columns="datetime")

features["any_error_24h"] = (features[[f"{e}_count_24h" for e in ERRORS]].sum(axis=1) > 0).astype(int)
print(features.shape, features.isna().sum().sum())

(36400, 17) 0


In [66]:
# Pick a machine-day that has errors, preferably several
r = features[features["any_error_24h"] == 1].sample(1, random_state=0).iloc[0]
T = r["prediction_time"]

e = err[(err.machineID == r["machineID"])
        & (err.datetime > T - pd.Timedelta(hours=24))
        & (err.datetime <= T)]

print(T, "machine", r["machineID"])
print(e["errorID"].value_counts())                       # counted by hand
print(r[[f"{x}_count_24h" for x in ERRORS]])             # the feature

2015-06-03 06:00:00 machine 90
errorID
error4    1
Name: count, dtype: int64
error1_count_24h    0.0
error2_count_24h    0.0
error3_count_24h    0.0
error4_count_24h    1.0
error5_count_24h    0.0
Name: 32548, dtype: object


In [67]:
maint = tables["maint"]
COMPS = ["comp1", "comp2", "comp3", "comp4"]

features = features.sort_values("prediction_time")
for comp in COMPS:
    last = (maint[maint["comp"] == comp][["machineID", "datetime"]]
              .drop_duplicates()
              .rename(columns={"datetime": "last_replaced"})
              .sort_values("last_replaced"))
    features = pd.merge_asof(features, last, left_on="prediction_time",
                             right_on="last_replaced", by="machineID", direction="backward")
    features[f"days_since_{comp}"] = (
        (features["prediction_time"] - features["last_replaced"]).dt.total_seconds() / 86400)
    features = features.drop(columns="last_replaced")
features = features.sort_values(["machineID", "prediction_time"]).reset_index(drop=True)

In [68]:
machines = tables["machines"]          # machineID, model, age

features = features.merge(machines, on="machineID", how="left")

# Models are categories (model1..model4), not numbers, so use one-hot columns
features = pd.get_dummies(features, columns=["model"], dtype=int)

print(features.shape)                  # expect (36400, 26): 21 + age + 4 model columns
print(features.isna().sum().sum())     # expect 0

(36400, 26)
0


In [69]:
print(len(features), features["failure_next_24h"].sum(), features["failure_next_24h"].mean().round(4))
print(features.duplicated(["machineID", "prediction_time"]).sum())    # expect 0, one row per machine-day

36400 699 0.0192
0


In [70]:
features.to_csv("../data/processed/machine_days.csv", index=False)

In [71]:
train = features[features["prediction_time"] <  "2015-09-01"]
val   = features[(features["prediction_time"] >= "2015-09-01") & (features["prediction_time"] < "2015-11-01")]
test  = features[features["prediction_time"] >= "2015-11-01"]

for name, part in [("train", train), ("val", val), ("test", test)]:
    print(name, len(part), part["failure_next_24h"].sum(), part["failure_next_24h"].mean().round(4))
    

train 24200 474 0.0196
val 6100 114 0.0187
test 6100 111 0.0182


In [72]:
def precision_recall(y, alert):
    tp = ((alert == 1) & (y == 1)).sum()
    fp = ((alert == 1) & (y == 0)).sum()
    fn = ((alert == 0) & (y == 1)).sum()
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    return round(precision, 3), round(recall, 3), int(tp + fp)   # last = number of alerts

In [73]:
y = val["failure_next_24h"]
print("error rule:", precision_recall(y, val["any_error_24h"]))

error rule: (np.float64(0.191), np.float64(1.0), 596)


In [74]:
SENSORS = ["volt", "rotate", "pressure", "vibration"]
tel_train = tel[tel["datetime"] < "2015-09-01"]
mu, sd = tel_train[SENSORS].mean(), tel_train[SENSORS].std()

def z(s):
    return (val[f"{s}_mean_24h"] - mu[s]) / sd[s]

sensor_alert = ((z("volt") > 1) | (z("rotate") < -1) |
                (z("pressure") > 1) | (z("vibration") > 1)).astype(int)

print("sensor rule:", precision_recall(y, sensor_alert))

sensor rule: (np.float64(0.149), np.float64(0.991), 756)


In [75]:
TARGET = "failure_next_24h"
DROP = ["machineID", "prediction_time", TARGET]
FEATURES = [c for c in features.columns if c not in DROP]

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]
print(len(FEATURES), "features")      # expect 25

23 features


In [76]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

logreg = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000))
gboost = HistGradientBoostingClassifier(random_state=0)

logreg.fit(X_train, y_train)
gboost.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",0
,"loss loss: {'log_loss'}, default='log_loss'The loss function to use in the boosting process.For binary classification problems, 'log_loss' is also known as logistic loss,binomial deviance or binary crossentropy. Internally, the model fits one treeper boosting iteration and uses the logistic sigmoid function (expit) asinverse link function to compute the predicted positive class probability.For multiclass classification problems, 'log_loss' is also known as multinomialdeviance or categorical crossentropy. Internally, the model fits one tree perboosting iteration and per class and uses the softmax function as inverse linkfunction to compute the predicted probabilities of the classes.",'log_loss'
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees for binary classification. For multiclassclassification, `n_classes` trees per iteration are built.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255
,"categorical_features categorical_features: array-like of {bool, int, str} of shape (n_features) or shape (n_categorical_features,), default='from_dtype'Indicates the categorical features.- None : no feature will be considered categorical.- boolean array-like : boolean mask indicating categorical features.- integer array-like : integer indices indicating categorical features.- str array-like: names of categorical features (assuming the training data has feature names).- `""from_dtype""`: dataframe columns with dtype ""Categorical"" and ""Enum"" are considered to be categorical features. The input must be a dataframe that is supported by narwhals (or supports it): :func:`narwhals.from_native` must work. This is the case, for instance, for pandas and polars DataFrames.For each categorical feature, there must be at most `max_bins` uniquecategories. Negative values for categorical features encoded as numericdtypes are treated as missing valu

In [77]:
from sklearn.metrics import average_precision_score, precision_recall_curve

def recall_at_precision(y, scores, floor=0.5):
    p, r, thr = precision_recall_curve(y, scores)
    ok = p[:-1] >= floor
    if not ok.any():
        return 0.0, None
    best = r[:-1][ok].argmax()
    return round(r[:-1][ok][best], 3), thr[ok][best]

for name, model in [("logreg", logreg), ("gboost", gboost)]:
    scores = model.predict_proba(X_val)[:, 1]
    print(name,
          "| PR-AUC:", round(average_precision_score(y_val, scores), 3),
          "| recall @ precision>=0.5:", recall_at_precision(y_val, scores))

logreg | PR-AUC: 0.864 | recall @ precision>=0.5: (np.float64(1.0), np.float64(0.19329639322906828))
gboost | PR-AUC: 1.0 | recall @ precision>=0.5: (np.float64(1.0), np.float64(5.621351869856428e-06))


In [78]:
import numpy as np

def operating_point(y, scores, floor=0.5):
    p, r, thr = precision_recall_curve(y, scores)
    ok = p[:-1] >= floor
    best_recall = r[:-1][ok].max()
    i = np.where(ok & (r[:-1] == best_recall))[0][-1]   # highest threshold with best recall
    return dict(threshold=round(thr[i], 4), precision=round(p[i], 3),
                recall=round(r[i], 3), alerts=int((scores >= thr[i]).sum()))

for name, model in [("logreg", logreg), ("gboost", gboost)]:
    print(name, operating_point(y_val, model.predict_proba(X_val)[:, 1]))

logreg {'threshold': np.float64(0.8449), 'precision': np.float64(0.712), 'recall': np.float64(1.0), 'alerts': 160}
gboost {'threshold': np.float64(0.8188), 'precision': np.float64(0.991), 'recall': np.float64(1.0), 'alerts': 115}


In [83]:
groups = {
    "sensors only":     [c for c in FEATURES if c.startswith(("volt", "rotate", "pressure", "vibration"))],
    "errors only":      [c for c in FEATURES if "error" in c],
    "sensors + errors": [c for c in FEATURES if c.startswith(("volt", "rotate", "pressure", "vibration")) or "error" in c],
    "sensors + errors + maintenance": [c for c in FEATURES if c.startswith(("volt", "rotate", "pressure", "vibration", "days_since")) or "error" in c],
    "all features":     FEATURES,
}

for name, cols in groups.items():
    m = HistGradientBoostingClassifier(random_state=0).fit(train[cols], y_train)
    s = m.predict_proba(val[cols])[:, 1]
    print(f"{name:18s} PR-AUC {average_precision_score(y_val, s):.3f}", operating_point(y_val, s))

sensors only       PR-AUC 0.195 {'threshold': np.float64(0.6994), 'precision': np.float64(0.5), 'recall': np.float64(0.018), 'alerts': 4}
errors only        PR-AUC 0.688 {'threshold': np.float64(0.2033), 'precision': np.float64(0.634), 'recall': np.float64(0.684), 'alerts': 123}
sensors + errors   PR-AUC 0.994 {'threshold': np.float64(0.0142), 'precision': np.float64(0.82), 'recall': np.float64(1.0), 'alerts': 139}
sensors + errors + maintenance PR-AUC 1.000 {'threshold': np.float64(0.6272), 'precision': np.float64(0.991), 'recall': np.float64(1.0), 'alerts': 115}
all features       PR-AUC 1.000 {'threshold': np.float64(0.8188), 'precision': np.float64(0.991), 'recall': np.float64(1.0), 'alerts': 115}


In [80]:
from sklearn.inspection import permutation_importance

imp = permutation_importance(gboost, X_val, y_val, scoring="average_precision",
                             n_repeats=5, random_state=0)
print(pd.Series(imp.importances_mean, index=FEATURES).sort_values(ascending=False).head(10))

any_error_24h         0.891789
volt_mean_24h         0.038846
vibration_mean_24h    0.031231
pressure_mean_24h     0.021680
rotate_mean_24h       0.020116
days_since_comp4      0.018567
days_since_comp2      0.015629
days_since_comp1      0.013536
days_since_comp3      0.012974
error5_count_24h      0.005913
dtype: float64


In [81]:
gate_tr = train["any_error_24h"] == 1
gate_va = val["any_error_24h"] == 1
print(gate_va.sum(), y_val[gate_va].sum())        # error-days in val, failures among them

cols_s = groups["sensors only"]
m = HistGradientBoostingClassifier(random_state=0).fit(train.loc[gate_tr, cols_s], y_train[gate_tr])
s = m.predict_proba(val.loc[gate_va, cols_s])[:, 1]
print("sensors on error-days only, PR-AUC:", round(average_precision_score(y_val[gate_va], s), 3))

596 114
sensors on error-days only, PR-AUC: 0.789


In [82]:
print(features.shape)
print([c for c in features.columns if "error" in c])

(36400, 26)
['error1_count_24h', 'error2_count_24h', 'error3_count_24h', 'error4_count_24h', 'error5_count_24h', 'any_error_24h']


In [84]:
chosen = "all features"                     # change to your pick, exactly as written in the groups dict
FEATURES = groups[chosen]
print(len(FEATURES), "features:", FEATURES)

23 features: ['volt_mean_24h', 'volt_std_24h', 'rotate_mean_24h', 'rotate_std_24h', 'pressure_mean_24h', 'pressure_std_24h', 'vibration_mean_24h', 'vibration_std_24h', 'error1_count_24h', 'error2_count_24h', 'error3_count_24h', 'error4_count_24h', 'error5_count_24h', 'any_error_24h', 'days_since_comp1', 'days_since_comp2', 'days_since_comp3', 'days_since_comp4', 'age', 'model_model1', 'model_model2', 'model_model3', 'model_model4']


In [85]:
X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]

gboost = HistGradientBoostingClassifier(random_state=0)
gboost.fit(X_train, y_train)

scores_val = gboost.predict_proba(X_val)[:, 1]
print("validation PR-AUC:", round(average_precision_score(y_val, scores_val), 3))

validation PR-AUC: 1.0


In [86]:
for t in [0.02, 0.05, 0.1, 0.2, 0.3, 0.5]:
    alert = (scores_val >= t).astype(int)
    print(t, precision_recall(y_val, alert))

0.02 (np.float64(0.974), np.float64(1.0), 117)
0.05 (np.float64(0.974), np.float64(1.0), 117)
0.1 (np.float64(0.974), np.float64(1.0), 117)
0.2 (np.float64(0.974), np.float64(1.0), 117)
0.3 (np.float64(0.974), np.float64(1.0), 117)
0.5 (np.float64(0.991), np.float64(1.0), 115)


In [87]:
ds = [c for c in FEATURES if c.startswith("days_since")]
m = HistGradientBoostingClassifier(random_state=0).fit(train[ds], y_train)
s = m.predict_proba(val[ds])[:, 1]
print("days_since only: PR-AUC", round(average_precision_score(y_val, s), 3))

days_since only: PR-AUC 0.312


In [88]:
print(val.groupby(TARGET)[ds].mean().round(1))

                  days_since_comp1  days_since_comp2  days_since_comp3  \
failure_next_24h                                                         
0                             42.8              36.9              40.4   
1                             43.5              46.1              49.3   

                  days_since_comp4  
failure_next_24h                    
0                             43.7  
1                             59.4  


In [89]:
tr2 = features[features["prediction_time"] < "2015-07-01"]
va2 = features[(features["prediction_time"] >= "2015-07-01") & (features["prediction_time"] < "2015-09-01")]
m2 = HistGradientBoostingClassifier(random_state=0).fit(tr2[FEATURES], tr2[TARGET])
s2 = m2.predict_proba(va2[FEATURES])[:, 1]
print("Jul–Aug PR-AUC:", round(average_precision_score(va2[TARGET], s2), 3))

Jul–Aug PR-AUC: 0.996


In [90]:
THRESHOLD = 0.2          # frozen from validation; do not change after this

X_test, y_test = test[FEATURES], test[TARGET]
scores_test = gboost.predict_proba(X_test)[:, 1]
alert_test = (scores_test >= THRESHOLD).astype(int)

print("gboost on test:    ", precision_recall(y_test, alert_test),
      "| PR-AUC:", round(average_precision_score(y_test, scores_test), 3))
print("error rule on test:", precision_recall(y_test, test["any_error_24h"]))
print("test rows:", len(test), "| failures:", int(y_test.sum()))

gboost on test:     (np.float64(0.965), np.float64(0.991), 114) | PR-AUC: 0.995
error rule on test: (np.float64(0.201), np.float64(1.0), 552)
test rows: 6100 | failures: 111
